# 07 · Circuit faithfulness

A candidate circuit is a hypothesis, not a conclusion. In this CPU-sized lesson the true pathway is known in advance, so we can ask whether a quantitative faithfulness test recovers it.

## Evidence target

We will test four quantities: the intact metric, the all-ablated null metric, performance with only the candidate retained, and performance with the candidate removed. The candidate must be both sufficient and necessary, then outperform equal-cardinality random circuits.

In [ ]:
import torch

from neuros_mechint.adapters import PyTorchAdapter
from neuros_mechint.benchmarks import (
    CircuitCandidate,
    FaithfulnessPolicy,
    GroundTruthCausalMLP,
    evaluate_adapter_circuit_faithfulness,
)
from neuros_mechint.core import OutputMetric


## Known mechanism

`GroundTruthCausalMLP` contains a serial `source -> causal` route and a nuisance route with zero contribution. Because the mechanism is known before analysis, this is a real falsification test rather than post-hoc storytelling.

In [ ]:
model = GroundTruthCausalMLP()
adapter = PyTorchAdapter(model)
inputs = torch.tensor([[1.0, 0.0]])
metric = OutputMetric(lambda output: output.mean(), name="mean_output")

candidate = CircuitCandidate(
    name="known-causal-route",
    targets=("source", "causal"),
)

report = evaluate_adapter_circuit_faithfulness(
    adapter=adapter,
    inputs=inputs,
    metric=metric,
    all_targets=("source", "causal", "nuisance"),
    candidate=candidate,
    random_trials=100,
    policy=FaithfulnessPolicy(
        min_sufficiency_fraction=0.99,
        min_necessity_fraction=0.99,
        min_random_percentile=0.99,
    ),
)
report.to_dict()


In [ ]:
assert report.sufficiency_fraction == 1.0
assert report.necessity_fraction == 1.0
assert report.joint_random_percentile == 1.0
assert report.passed

[(control.targets, control.joint_faithfulness) for control in report.random_controls]


## Why the joint random score matters

In serial pathways, many incorrect circuits can look necessary because removing almost any upstream/downstream combination destroys the output. They are still not sufficient. v0.5 therefore ranks random controls by `min(sufficiency, necessity)` while retaining both component scores for diagnosis.

## Transfer to external tools

Replace `PyTorchAdapter` with a `TransformerLensAdapter` or `NNsightAdapter` and keep the same faithfulness benchmark. For SAEs, use `evaluate_sae_feature_faithfulness`, which records the SAE reconstruction gap before scoring feature subsets. A circuit-tracer graph may nominate a `CircuitCandidate`, but the attribution graph itself is not the faithfulness result.

## Falsification checklist

Before promoting a real circuit claim: repeat on held-out examples, vary the ablation baseline, compare equal-cardinality random circuits, test multiple seeds/checkpoints, and report failures. Passing this notebook demonstrates the benchmark on a known mechanism; it does not validate an unknown-model circuit.